# Component 1: ResNet18 Vision Encoder

In diffusion policy, the robot needs to **see** its environment. Raw pixels are too high-dimensional to work with directly (a single 96x96 RGB image has 27,648 values), so we compress them into a compact feature vector using a **convolutional neural network**.

The architecture used is **ResNet18** -- a well-known image classification backbone with 18 layers organized into 4 residual blocks. Each block doubles the number of feature channels while halving the spatial resolution. The "residual" in ResNet refers to skip connections that add the input of each block to its output, making training much more stable.

In this notebook we will:
1. Extract the ResNet18 encoder from a trained diffusion policy
2. Trace an image through each layer and observe how shapes change
3. Visualize what the network "sees" at different depths
4. Plot activation statistics across layers

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import torchvision.models as models
import torch.nn as nn

# Load a pretrained ResNet18 -- the same backbone used in diffusion policy
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# In diffusion policy, we remove the final FC layer and avgpool.
# The encoder uses everything up to and including layer4.
# Let's keep the full backbone for inspection.

print("ResNet18 loaded with ImageNet pretrained weights!")
print(f"Total parameters: {sum(p.numel() for p in backbone.parameters()):,}")

# For diffusion policy, the input is 96x96 (not 224x224 like ImageNet).
# Let's create a test image at that resolution.
# We'll use a colorful synthetic image so we can see meaningful feature maps.

torch.manual_seed(42)
# Create a synthetic "scene" with distinct regions (simulates a robot workspace)
test_img = torch.zeros(1, 3, 96, 96)
# Red region (top-left) -- like a red cube on the table
test_img[0, 0, 10:40, 10:40] = 0.9
test_img[0, 1, 10:40, 10:40] = 0.2
test_img[0, 2, 10:40, 10:40] = 0.2
# Blue region (bottom-right) -- like a blue target zone
test_img[0, 0, 55:85, 55:85] = 0.2
test_img[0, 1, 55:85, 55:85] = 0.3
test_img[0, 2, 55:85, 55:85] = 0.9
# Green background
test_img[0, 1, :, :] += 0.15
# Add some noise for realism
test_img += torch.randn_like(test_img) * 0.05
test_img = test_img.clamp(0, 1)

# Apply ImageNet normalization (same as diffusion policy preprocessing)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
test_img_normalized = (test_img - MEAN) / STD

test_img_normalized = test_img_normalized.to(device)
backbone = backbone.to(device)
backbone.eval()

print(f"\nTest image shape: {test_img.shape}")
print(f"  -> Batch=1, Channels=3 (RGB), Height=96, Width=96")
print(f"  -> Total input values: {3*96*96:,}")

# Extracting the ResNet18 Backbone

## How Diffusion Policy Uses ResNet18

In the diffusion policy codebase, the vision encoder wraps a standard torchvision ResNet18:
- It removes the final `avgpool` and `fc` layers (we don't want ImageNet class predictions)
- It keeps everything through `layer4`, producing a **512-channel feature map**
- For a 96×96 input, the output is a **512×3×3 feature map** (4,608 values)

Let's trace our test image through each layer to see exactly how the shapes change.

In [ ]:
print("=" * 70)
print("ResNet18 Backbone Architecture (layers used by diffusion policy)")
print("=" * 70)

# Print each major component
for name, module in backbone.named_children():
    if name in ['avgpool', 'fc']:
        print(f"  {name:15s} <- REMOVED in diffusion policy")
    else:
        num_params = sum(p.numel() for p in module.parameters())
        print(f"  {name:15s} ({num_params:>10,} params)")

print("=" * 70)
total = sum(p.numel() for p in backbone.parameters())
print(f"Total parameters: {total:,}")
print(f"(In diffusion policy, these are initialized from ImageNet weights)")

## Tracing an Image Through Each Layer

We will pass a 96x96 test image through the backbone one layer at a time and record the output shape at each stage. This shows how the spatial resolution shrinks and the channel count grows as we go deeper.

In [ ]:
print(f"Input image shape: {test_img_normalized.shape}")
print(f"  -> Batch=1, Channels=3 (RGB), Height=96, Width=96\n")

# Trace through each major layer
layer_names = []
layer_outputs = []

with torch.no_grad():
    x = test_img_normalized

    # Initial convolution + batchnorm + relu + maxpool
    x = backbone.conv1(x)
    x = backbone.bn1(x)
    x = backbone.relu(x)
    layer_names.append("conv1 + bn + relu")
    layer_outputs.append(x.clone())
    print(f"After conv1 + bn + relu:  {x.shape}")

    x = backbone.maxpool(x)
    layer_names.append("maxpool")
    layer_outputs.append(x.clone())
    print(f"After maxpool:            {x.shape}")

    # Residual blocks
    x = backbone.layer1(x)
    layer_names.append("layer1 (block 1)")
    layer_outputs.append(x.clone())
    print(f"After layer1 (block 1):   {x.shape}")

    x = backbone.layer2(x)
    layer_names.append("layer2 (block 2)")
    layer_outputs.append(x.clone())
    print(f"After layer2 (block 2):   {x.shape}")

    x = backbone.layer3(x)
    layer_names.append("layer3 (block 3)")
    layer_outputs.append(x.clone())
    print(f"After layer3 (block 3):   {x.shape}")

    x = backbone.layer4(x)
    layer_names.append("layer4 (block 4)")
    layer_outputs.append(x.clone())
    print(f"After layer4 (block 4):   {x.shape}")

print("\n--- Summary ---")
print(f"Input:  3 channels x 96 x 96  = {3*96*96:,} values")
print(f"Output: {x.shape[1]} channels x {x.shape[2]} x {x.shape[3]} = {x.shape[1]*x.shape[2]*x.shape[3]:,} values")
print(f"\nCompression ratio: {3*96*96 / (x.shape[1]*x.shape[2]*x.shape[3]):.1f}x")
print("The network compressed 27,648 pixel values into 4,608 feature values!")

In [ ]:
# Visualize 8 feature map channels from different layers (2x4 grid per layer)
layers_to_show = [
    ("conv1 + bn + relu", layer_outputs[0]),
    ("layer1 (block 1)", layer_outputs[2]),
    ("layer2 (block 2)", layer_outputs[3]),
    ("layer4 (block 4)", layer_outputs[5]),
]

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle("Feature Maps at Different Layers (8 channels each)", fontsize=16, fontweight="bold")

for row, (name, feats) in enumerate(layers_to_show):
    feat_np = feats[0].cpu().numpy()  # (C, H, W)
    num_channels = min(8, feat_np.shape[0])
    for col in range(8):
        ax = axes[row, col]
        if col < num_channels:
            ch_idx = col * (feat_np.shape[0] // 8)  # spread across channels
            im = ax.imshow(feat_np[ch_idx], cmap="viridis", aspect="auto")
            if col == 0:
                ax.set_ylabel(name, fontsize=9, fontweight="bold")
            ax.set_title(f"ch {ch_idx}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

# Also show the original image for reference
fig2, ax2 = plt.subplots(1, 1, figsize=(3, 3))
img_display = test_img[0].permute(1, 2, 0).numpy()  # unnormalized version
ax2.imshow(img_display)
ax2.set_title("Input Image (96x96)\n(synthetic robot workspace)")
ax2.axis("off")
plt.show()

In [ ]:
# Plot activation statistics (mean, std) per layer
means = [out.mean().item() for out in layer_outputs]
stds = [out.std().item() for out in layer_outputs]
maxs = [out.max().item() for out in layer_outputs]
spatial_sizes = [f"{out.shape[2]}x{out.shape[3]}" for out in layer_outputs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_pos = range(len(layer_names))

# Mean and Std
ax1.bar(x_pos, means, width=0.4, label="Mean", color="#5B8CB8", align="center")
ax1.bar([x + 0.4 for x in x_pos], stds, width=0.4, label="Std Dev", color="#D97757", align="center")
ax1.set_xticks([x + 0.2 for x in x_pos])
ax1.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Value")
ax1.set_title("Activation Statistics per Layer")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Spatial size reduction
channels = [out.shape[1] for out in layer_outputs]
spatial = [out.shape[2] * out.shape[3] for out in layer_outputs]

ax2_twin = ax2.twinx()
ax2.bar(x_pos, channels, width=0.4, color="#7DA488", alpha=0.8, label="Channels")
ax2_twin.plot(x_pos, spatial, "o-", color="#9B7EC8", linewidth=2, markersize=8, label="Spatial (HxW)")

ax2.set_xticks(x_pos)
ax2.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("Number of Channels", color="#7DA488")
ax2_twin.set_ylabel("Spatial Dimensions (HxW)", color="#9B7EC8")
ax2.set_title("Channels vs Spatial Resolution")

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="center right")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight: As we go deeper, spatial resolution shrinks (96->3)")
print("while channel count grows (64->512). The network trades spatial")
print("detail for semantic richness.")

## TODO Exercise

**What happens if you use a 224x224 image instead of 96x96?**

1. Before running any code, calculate the expected output shape at each layer. Remember that:
   - `conv1` uses kernel=7, stride=2, padding=3 (halves spatial dims)
   - `maxpool` uses kernel=3, stride=2, padding=1 (halves again)
   - `layer2`, `layer3`, `layer4` each halve spatial dims

2. Write your predictions here, then verify:

```python
# TODO: Create a 224x224 test image and trace it through the backbone
# test_224 = torch.randn(1, 3, 224, 224).to(device)
# ... trace through each layer and print shapes ...
# Compare with your predictions!
```

3. What does this mean for the SpatialSoftmax layer that comes next? (Hint: more spatial locations = more precise keypoints)